# Volatility Regime Analysis — Does the Model Learn Market Dynamics?

Investigates whether the generative model captures **period-specific volatility**:
do generated samples conditioned on a historically volatile time window exhibit
higher variance than samples from a calm period?

### Method
1. Build the full reference window corpus (`SP500WindowDataset`) and compute a
   **volatility proxy** (std of 512 Close log-returns) for each of the 25 k windows.
2. Map every trading date to the distribution of window-stds that cover it,
   producing a time-resolved **mean volatility signal** for each date.
3. Load generated samples; for each generated window, compute its
   **reference-period volatility** by averaging the date-level signal over its
   date range.
4. Classify generated windows into **low / medium / high** volatility regimes
   using reference tertiles, then test whether the model's generated std
   tracks those regimes.

In [12]:
# ── USER INPUTS ───────────────────────────────────────────────────────────────
CHECKPOINT_STEM = (
    "csv20000_samples5_steps200_seed50_20260604_233534_UNCO_Pm-1.4_Ps1.8"
)  # sub-folder under GEN_DIR produced by generate_samples.py

GEN_DIR  = "../../Master-Thesis/data/generated/final/edm"
SPLIT    = "val"   # "train" | "val" | "both"

# Reference dataset — full sliding-window corpus
REFERENCE_ROOT_DIR = "../../Master-Thesis/data/SNP500_individual_normalized_replication"
SEQ_LEN  = 512
STRIDE   = 100
COLUMNS  = ("date", "close", "open", "high", "low")
CLOSE_FEATURE_IDX = 0   # index of Close in the feature (K) axis
DATE_FORMAT = "%d/%m/%Y"  # format used in the CSV files

# Normalization stats — for unnormalizing log-returns
STATS_FILE = "../../Master-Thesis/data/general/normalization_stats_norm_replication.csv"

# Regime definition: tertile thresholds on reference period-volatility
# Fractions (0–1) separating low / medium / high
REGIME_LOW_Q  = 1/3
REGIME_HIGH_Q = 2/3

# Image output
OUTPUT_DIR = "../../Master-Thesis/images/final/edm/volatility_regime"
appendix   = "rep_Pm-1.4_Ps1.8_UNCO_vol_regime_val"  # string appended to output filenames to distinguish checkpoints
# ─────────────────────────────────────────────────────────────────────────────

## 0. Imports

In [13]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as scipy_stats

sys.path.insert(0, "..")
sys.path.insert(0, "../../Master-Thesis")

from torch.utils.data import DataLoader
from src.utils.dataloader import SP500WindowDataset, csdi_collate_fn

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

Output directory: c:\Users\Lenovo\Documents\SCUOLA\UNI\MASTER\2ANNO\THESIS\Master-Thesis\images\final\edm\volatility_regime


## 1. Load Reference Corpus with Per-Window Dates

Instantiates `SP500WindowDataset` and iterates directly over every window
(bypassing the DataLoader) so that the 512 trading-date strings per window
are accessible alongside the Close log-return values.

`cache_data=True` ensures each CSV file is parsed only once; subsequent
windows from the same file are served from RAM.

**Output**
- `ref_close` : `(N_ref, 512)` — normalized Close log-returns
- `ref_dates` : `(N_ref, 512)` — trading-date strings for each step

In [14]:
print("Building SP500WindowDataset …")
ref_dataset = SP500WindowDataset(
    root_dir=REFERENCE_ROOT_DIR,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    columns=COLUMNS,
    time_mode="global_index",
    cache_data=True,
    drop_incomplete=True,
)
n_ref = len(ref_dataset)
print(f"  {n_ref:,} windows across {len(ref_dataset.files)} files")

# Sorted global trading calendar (date_str → int index)
global_calendar = sorted(
    ref_dataset.date_to_idx.keys(),
    key=lambda s: pd.to_datetime(s, format=DATE_FORMAT)
)
calendar_set = set(global_calendar)
# Map date_str → pandas Timestamp for later comparisons
date_to_ts = {d: pd.to_datetime(d, format=DATE_FORMAT) for d in global_calendar}
print(f"  Global calendar: {len(global_calendar):,} unique trading dates "
      f"({date_to_ts[global_calendar[0]].date()} → {date_to_ts[global_calendar[-1]].date()})")

# ── Collect close series + dates for every window ────────────────────────────
# Iterates directly (not via DataLoader) to access internal _load_file cache.
ref_close_list = []
ref_dates_list = []

for i, (file_idx, start) in enumerate(ref_dataset.index):
    fp = ref_dataset.files[file_idx]
    x, dates = ref_dataset._load_file(fp)  # (T, K), (T,) — cached after first call
    end = start + SEQ_LEN
    ref_close_list.append(x[start:end, CLOSE_FEATURE_IDX])  # (L,)
    ref_dates_list.append(dates[start:end])                  # (L,)

ref_close = np.array(ref_close_list, dtype=np.float32)  # (N_ref, L)
ref_dates  = np.array(ref_dates_list, dtype=object)      # (N_ref, L) — str arrays

print(f"\nref_close : {ref_close.shape}")
print(f"ref_dates : {ref_dates.shape}")
print(f"Sample window dates: {ref_dates[0, 0]}  →  {ref_dates[0, -1]}")

Building SP500WindowDataset …
  25,226 windows across 210 files
  Global calendar: 16,175 unique trading dates (1962-01-03 → 2026-04-10)

ref_close : (25226, 512)
ref_dates : (25226, 512)
Sample window dates: 15/12/1980  →  22/12/1982


## 2. Normalization Stats and Unnormalization Helper

In [15]:
stats_df = pd.read_csv(STATS_FILE, header=0, index_col=0)
stats_df.columns = ["mean", "std"]
stats_df.index   = stats_df.index.str.strip().str.lower()
col_stats = stats_df.to_dict(orient="index")

print("Normalization statistics:")
display(stats_df)

def unnorm(arr, feature="close"):
    mu  = col_stats[feature]["mean"]
    sig = col_stats[feature]["std"]
    return arr * sig + mu

ref_close_un = unnorm(ref_close)   # (N_ref, L) — original log-return scale

Normalization statistics:


,mean,std
close,0.000451,0.020867
open,0.000265,0.010859
high,0.012179,0.019033
low,-0.011306,0.017881


## 3. Per-Window Volatility (Std of Close Log-Returns)

For each reference window a single volatility scalar is computed as
`std(close_512_steps)`.  Both normalized and unnormalized versions are kept;
the **unnormalized** std is the primary metric because it corresponds to real
percentage-point log-return variability and is directly comparable across models.

In [16]:
ref_std_norm = ref_close.std(axis=1)     # (N_ref,) — normalized space
ref_std_un   = ref_close_un.std(axis=1)  # (N_ref,) — unnormalized (log-return scale)

print(f"Reference window std (unnorm) — summary:")
df_ref_std = pd.Series(ref_std_un).describe(percentiles=[.05, .25, .50, .75, .95])
display(df_ref_std.to_frame("ref_std_unnorm").T.round(6))

# For the regime definition use unnormalized std so the percentile thresholds
# are in interpretable log-return units.
q_low  = np.quantile(ref_std_un, REGIME_LOW_Q)
q_high = np.quantile(ref_std_un, REGIME_HIGH_Q)
print(f"\nRegime thresholds (unnorm std):")
print(f"  Low  / Medium boundary : {q_low:.6f}")
print(f"  Medium / High boundary : {q_high:.6f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(ref_std_un, bins=100, color="tab:blue", alpha=0.7)
ax.axvline(q_low,  color="orange", lw=1.5, ls="--", label=f"Low→Med ({q_low:.4f})")
ax.axvline(q_high, color="red",    lw=1.5, ls="--", label=f"Med→High ({q_high:.4f})")
ax.set_xlabel("Window std (unnorm log-return)")
ax.set_ylabel("Count")
ax.set_title("Distribution of per-window volatility across reference corpus")
ax.legend(fontsize=9)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"ref_window_std_dist_{appendix}.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

Reference window std (unnorm) — summary:


,count,mean,std,min,5%,25%,50%,75%,95%,max
ref_std_unnorm,25226.0,0.019006,0.008562,0.004755,0.009828,0.013341,0.017026,0.022308,0.034802,0.107888



Regime thresholds (unnorm std):
  Low  / Medium boundary : 0.014486
  Medium / High boundary : 0.020251


<Figure size 800x300 with 1 Axes>

## 4. Date → Volatility Mapping

Every trading date that appears in a reference window is associated with
that window's std.  Because windows slide with stride 100 and overlap
substantially, each date appears in many windows.  The per-date statistics
are:

- `date_mean_std` — average window-std across all windows covering that date
- `date_std_of_std` — dispersion of those stds (how consistently volatile)

The resulting **time series of mean volatility** is the key reference signal.

In [17]:
date_to_stds_un = defaultdict(list)   # date_str -> [std_un, ...]

for i in range(n_ref):
    std_i = float(ref_std_un[i])
    for d in ref_dates[i]:
        date_to_stds_un[d].append(std_i)

dates_sorted  = sorted(date_to_stds_un.keys(), key=lambda s: pd.to_datetime(s, format=DATE_FORMAT))
ts_sorted     = [pd.to_datetime(d, format=DATE_FORMAT) for d in dates_sorted]
mean_std_ts   = np.array([np.mean(date_to_stds_un[d])  for d in dates_sorted])
std_of_std_ts = np.array([np.std(date_to_stds_un[d])   for d in dates_sorted])
n_windows_ts  = np.array([len(date_to_stds_un[d])      for d in dates_sorted])

# Build lookup dict for fast access later
date_mean_std = dict(zip(dates_sorted, mean_std_ts))  # date_str -> float

print(f"Unique dates with volatility signal : {len(dates_sorted):,}")
print(f"Avg windows per date               : {n_windows_ts.mean():.1f}")
print(f"Top-5 most volatile dates (mean std):")
top5_idx = np.argsort(mean_std_ts)[-5:][::-1]
for idx in top5_idx:
    print(f"  {ts_sorted[idx].date()}  mean_std={mean_std_ts[idx]:.5f}  (n_windows={n_windows_ts[idx]})")

Unique dates with volatility signal : 16,175
Avg windows per date               : 798.5
Top-5 most volatile dates (mean std):
  2008-11-05  mean_std=0.03082  (n_windows=1053)
  2008-11-04  mean_std=0.03081  (n_windows=1053)
  2008-11-06  mean_std=0.03081  (n_windows=1054)
  2008-11-07  mean_std=0.03080  (n_windows=1055)
  2008-11-10  mean_std=0.03080  (n_windows=1055)


## 5. Time-Series of Reference Volatility

Plots the mean-std signal over the full calendar.  High peaks correspond
to known crises (Dot-com 2000–2001, GFC 2008–2009, COVID 2020) — this
sanity-check validates that the proxy captures real market dynamics.

In [18]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax = axes[0]
ax.plot(ts_sorted, mean_std_ts, lw=0.8, color="tab:blue", label="Mean std per date")
ax.fill_between(
    ts_sorted,
    mean_std_ts - std_of_std_ts,
    mean_std_ts + std_of_std_ts,
    alpha=0.25, color="tab:blue", label="±1 std-of-std"
)
ax.set_ylabel("Avg window std\n(unnorm log-return)")
ax.set_title("Reference corpus: date-level mean volatility signal")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(ts_sorted, n_windows_ts, lw=0.6, color="grey")
ax.set_ylabel("Windows covering date")
ax.set_xlabel("Date")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"ref_volatility_timeseries_{appendix}.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

<Figure size 1400x600 with 2 Axes>

## 6. Load Generated Samples

Reads the generated-close CSV for the selected split.  Each row is one
*(window, sample)* pair.  `start_date` / `end_date` columns identify the
calendar period of each generated window.

If the CSV was produced without date columns (only `window_idx` /
`sample_idx`), the script raises an informative error — date columns are
required to map to the reference volatility signal.

In [19]:
ckpt_dir  = os.path.join(GEN_DIR, CHECKPOINT_STEM)
splits_to_run = ["train", "val"] if SPLIT == "both" else [SPLIT]

gen_frames = []
for split in splits_to_run:
    gen_path = os.path.join(ckpt_dir, f"{split}_generated_close.csv")
    assert os.path.isfile(gen_path), f"Not found: {gen_path}"
    df = pd.read_csv(gen_path)
    df["_split"] = split
    gen_frames.append(df)

df_gen = pd.concat(gen_frames, ignore_index=True)
step_cols = [c for c in df_gen.columns if c.startswith("step_")]

print("Generated CSV columns:", list(df_gen.columns[:10]), "... (+step columns)")
print(f"Rows: {len(df_gen):,}   Step columns: {len(step_cols)}")

# ── Validate required columns ─────────────────────────────────────────────────
# The CSV schema uses 'file' (source CSV basename) and 'window_start'
# (integer row index within that file).  Together they uniquely identify which
# 512 trading dates the window covers, without needing explicit date columns.
required = {"file", "window_start"}
missing  = required - set(df_gen.columns)
if missing:
    raise ValueError(
        f"Generated CSV is missing required columns: {missing}. Expected columns: window_idx, sample_idx, file, window_start, step_000 … step_511.")

df_gen["window_start"] = df_gen["window_start"].astype(int)

n_gen_windows = df_gen[["file", "window_start"]].drop_duplicates().shape[0]
n_gen_samples = df_gen["sample_idx"].nunique() if "sample_idx" in df_gen.columns else 1
print(f"Unique (file, window_start) windows : {n_gen_windows}")
print(f"Samples per window                  : {n_gen_samples}")
print(df_gen[["file", "window_start"]].drop_duplicates().head())

Generated CSV columns: ['window_idx', 'sample_idx', 'file', 'window_start', 'step_000', 'step_001', 'step_002', 'step_003', 'step_004', 'step_005'] ... (+step columns)
Rows: 1,185   Step columns: 512
Unique (file, window_start) windows : 237
Samples per window                  : 5
                             file  window_start
0   ABT_1980-03-17_2026-04-10.csv          8700
5   ADI_1980-03-17_2026-04-10.csv          7400
10  ADP_1980-03-17_2026-04-10.csv          5100
15  AEP_1962-01-02_2026-04-10.csv          1800
20  AEP_1962-01-02_2026-04-10.csv          1900


## 6b. Reconstruct Exact Dates for Each Generated Window

Since the CSV provides `file` (source stock CSV) and `window_start` (integer
row index), we can retrieve the exact same 512 trading-date strings that
`SP500WindowDataset` would have served for this window — with no interpolation.

`_load_file` is already cached from § 1, so this loop is just array slicing.

**Output**
- `win_dates` : dict `(file, window_start)` → `np.ndarray` of 512 date strings
- `df_gen["start_date_ts"]` : first date of each row's window (for time-series plots)

In [20]:
# Build a lookup: (file_basename, window_start) -> 512 date strings
# ref_dataset._load_file is already populated in the cache from § 1,
# so these calls just return the in-memory arrays.
win_dates: dict = {}   # (file, window_start) -> np.ndarray(512,) of date strings

unique_wins = df_gen[["file", "window_start"]].drop_duplicates().values

for fname, w_start in unique_wins:
    full_path = os.path.join(REFERENCE_ROOT_DIR, fname)
    if not os.path.isfile(full_path):
        print(f"[WARN] Source file not found in reference dir: {fname} — skipping")
        win_dates[(fname, w_start)] = None
        continue
    _, dates_file = ref_dataset._load_file(full_path)  # (T,) — cached
    w_end = w_start + SEQ_LEN
    if w_end > len(dates_file):
        print(f"[WARN] window_start={w_start} + SEQ_LEN={SEQ_LEN} > file length {len(dates_file)} for {fname}")
        win_dates[(fname, w_start)] = None
        continue
    win_dates[(fname, w_start)] = dates_file[w_start:w_end]

# Attach first date of each window as a Timestamp for plot x-axis
def first_date_ts(row):
    key   = (row["file"], row["window_start"])
    dates = win_dates.get(key)
    if dates is None or len(dates) == 0:
        return pd.NaT
    return pd.to_datetime(dates[0], format=DATE_FORMAT)

df_gen["start_date_ts"] = df_gen.apply(first_date_ts, axis=1)

n_ok = sum(1 for v in win_dates.values() if v is not None)
print(f"Date reconstruction: {n_ok} / {len(win_dates)} windows succeeded")
print(f"Generated window date range: "
      f"{df_gen['start_date_ts'].min().date()} → {df_gen['start_date_ts'].max().date()}")

[WARN] Source file not found in reference dir: BK_1973-05-03_2026-04-10.csv — skipping
[WARN] Source file not found in reference dir: BK_1973-05-03_2026-04-10.csv — skipping
[WARN] Source file not found in reference dir: HUBB_1972-06-05_2026-04-10.csv — skipping
[WARN] Source file not found in reference dir: HUBB_1972-06-05_2026-04-10.csv — skipping
[WARN] Source file not found in reference dir: WMB_1981-12-31_2026-04-10.csv — skipping
Date reconstruction: 232 / 237 windows succeeded
Generated window date range: 1966-10-06 → 2024-01-22


## 7. Assign Reference-Period Volatility to Each Generated Window

For each generated window with known `start_date` and `end_date`:
1. Find all global calendar dates in `[start_date, end_date]`.
2. Look up their `date_mean_std` from the reference corpus.
3. Average → **period_ref_std**: a single scalar summarising how volatile
   the reference market was during that window's time span.

Generated windows are then classified into **Low / Medium / High** regimes
using the tertile thresholds computed on the full reference corpus.  This
avoids any data leakage from the generated samples themselves into the
regime boundaries.

In [21]:
def period_ref_volatility(fname: str, w_start: int) -> float:
    """Average date_mean_std over the 512 dates of window (fname, w_start)."""
    dates = win_dates.get((fname, w_start))
    if dates is None:
        return np.nan
    vals = [date_mean_std[d] for d in dates if d in date_mean_std]
    return float(np.mean(vals)) if vals else np.nan


def assign_regime(v: float) -> str:
    if np.isnan(v):
        return "Unknown"
    if v < q_low:
        return "Low"
    elif v < q_high:
        return "Medium"
    else:
        return "High"


# Compute per-row (all rows sharing the same window get the same value)
df_gen["period_ref_std"] = [
    period_ref_volatility(row["file"], row["window_start"])
    for _, row in df_gen.iterrows()
]
df_gen["regime"] = df_gen["period_ref_std"].map(assign_regime)

regime_counts = df_gen["regime"].value_counts()
print("Rows per regime (all samples):")
display(regime_counts.to_frame())

print(f"\nUnique windows per regime:")
display(
    df_gen.groupby(["file", "window_start"])["regime"]
    .first()
    .value_counts()
    .to_frame()
)

Rows per regime (all samples):


,count
regime,
Medium,705
High,305
Low,150
Unknown,25



Unique windows per regime:


,count
regime,
Medium,141
High,61
Low,30
Unknown,5


## 8. Compute Generated Window Volatility

For each generated window (identified by `start_date` + `end_date`):
- Compute `std` of the 512-step Close series **per sample**.
- Report the **mean across samples** as the window-level generated volatility.

Both normalized and unnormalized versions are computed.  Unnormalized
stds are directly comparable to the reference `period_ref_std`.

In [22]:
gen_arr    = df_gen[step_cols].to_numpy(dtype=np.float32)   # (N_rows, 512)
gen_arr_un = unnorm(gen_arr)                                # unnormalized

df_gen["gen_std_norm"] = gen_arr.std(axis=1)               # per sample
df_gen["gen_std_un"]   = gen_arr_un.std(axis=1)

# Aggregate to window level: mean std across samples
group_key = ["file", "window_start", "regime", "period_ref_std", "start_date_ts"]
df_win_agg = (
    df_gen
    .groupby(group_key, observed=True)[["gen_std_norm", "gen_std_un"]]
    .mean()
    .reset_index()
)

print("Window-level generated volatility summary by regime (unnorm std):")
display(
    df_win_agg.groupby("regime")["gen_std_un"]
    .describe(percentiles=[.25, .50, .75])
    .round(6)
)

Window-level generated volatility summary by regime (unnorm std):


,count,mean,std,min,25%,50%,75%,max
regime,,,,,,,,
High,61.0,0.016948,0.002947,0.011604,0.015227,0.016717,0.018810,0.026044
Low,30.0,0.012119,0.002037,0.009411,0.011331,0.011945,0.012501,0.021069
Medium,141.0,0.014022,0.002236,0.009598,0.012525,0.013857,0.015030,0.022310


## 9. Comparison: Generated Volatility by Regime

If the model has learned period-specific volatility, windows from the **High**
regime should produce samples with higher std than **Low** regime windows.

Three complementary views:
1. **Box plot** of generated_std per regime (visual).
2. **Scatter plot** of `period_ref_std` vs `gen_std_un` (correlation).
3. **Statistical tests** (Kruskal-Wallis + pairwise Mann-Whitney U).

In [23]:
REGIME_ORDER  = ["Low", "Medium", "High"]
REGIME_COLORS = {"Low": "tab:blue", "Medium": "tab:orange", "High": "tab:red"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── 1. Box plot ───────────────────────────────────────────────────────────────
ax = axes[0]
data_by_regime = [
    df_win_agg.loc[df_win_agg["regime"] == r, "gen_std_un"].dropna().values
    for r in REGIME_ORDER
    if r in df_win_agg["regime"].values
]
labels_present = [r for r in REGIME_ORDER if r in df_win_agg["regime"].values]
bp = ax.boxplot(data_by_regime, labels=labels_present, patch_artist=True, notch=False)
for patch, label in zip(bp["boxes"], labels_present):
    patch.set_facecolor(REGIME_COLORS[label])
    patch.set_alpha(0.6)
ax.set_ylabel("Mean gen std (unnorm log-return)")
ax.set_title("Generated window volatility by regime")
ax.set_xlabel("Reference volatility regime")

# ── 2. Scatter: period_ref_std vs gen_std_un ─────────────────────────────────
ax = axes[1]
for regime, grp in df_win_agg.groupby("regime", observed=True):
    ax.scatter(
        grp["period_ref_std"], grp["gen_std_un"],
        s=20, alpha=0.6, label=regime,
        color=REGIME_COLORS.get(regime, "grey")
    )
# OLS trend line
valid = df_win_agg[["period_ref_std", "gen_std_un"]].dropna()
if len(valid) > 2:
    slope, intercept, r, p, _ = scipy_stats.linregress(valid["period_ref_std"], valid["gen_std_un"])
    x_line = np.linspace(valid["period_ref_std"].min(), valid["period_ref_std"].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, "k--", lw=1.2,
            label=f"OLS  r={r:.3f}  p={p:.3e}")
ax.set_xlabel("Reference period std (unnorm)")
ax.set_ylabel("Generated mean std (unnorm)")
ax.set_title("Scatter: reference vs generated volatility")
ax.legend(fontsize=8)

# ── 3. KDE of generated std per regime ───────────────────────────────────────
ax = axes[2]
for regime in labels_present:
    vals = df_win_agg.loc[df_win_agg["regime"] == regime, "gen_std_un"].dropna().values
    if len(vals) < 2:
        continue
    kde = scipy_stats.gaussian_kde(vals)
    x_kde = np.linspace(vals.min() * 0.8, vals.max() * 1.2, 300)
    ax.plot(x_kde, kde(x_kde), color=REGIME_COLORS[regime], lw=2, label=regime)
    ax.axvline(vals.mean(), color=REGIME_COLORS[regime], lw=1, ls=":")
ax.set_xlabel("Generated mean std (unnorm log-return)")
ax.set_ylabel("Density")
ax.set_title("KDE of generated std per regime")
ax.legend(fontsize=9)

plt.suptitle(
    "Does the model generate higher volatility for historically volatile periods?",
    y=1.02, fontsize=12
)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"regime_comparison_{appendix}.png"), dpi=250, bbox_inches="tight")
plt.close(fig)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27152\1190226318.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_by_regime, labels=labels_present, patch_artist=True, notch=False)


<Figure size 1600x500 with 3 Axes>

## 10. Statistical Tests

- **Kruskal-Wallis H-test**: non-parametric one-way ANOVA across all three regimes.
  Significant result means at least one group differs.
- **Pairwise Mann-Whitney U**: which pairs differ; p-values Bonferroni-corrected
  for 3 comparisons.

A significant Low vs High difference at the correct direction (High generates
larger std) would support the hypothesis that the model captures period volatility.

In [24]:
groups = {
    r: df_win_agg.loc[df_win_agg["regime"] == r, "gen_std_un"].dropna().values
    for r in REGIME_ORDER
    if r in df_win_agg["regime"].values
}
print("Group sizes:", {k: len(v) for k, v in groups.items()})

# ── Kruskal-Wallis ────────────────────────────────────────────────────────────
if len(groups) >= 2 and all(len(v) >= 5 for v in groups.values()):
    kw_stat, kw_p = scipy_stats.kruskal(*groups.values())
    print(f"\nKruskal-Wallis H = {kw_stat:.4f}   p = {kw_p:.4e}")
    if kw_p < 0.05:
        print("→ Significant difference across regimes (p < 0.05)")
    else:
        print("→ No significant difference across regimes (p ≥ 0.05)")
else:
    print("Skipping Kruskal-Wallis: not enough groups or samples.")

# ── Pairwise Mann-Whitney U (Bonferroni n=3) ──────────────────────────────────
print("\nPairwise Mann-Whitney U (Bonferroni-corrected α = 0.05/3):")
pairs = [("Low", "High"), ("Low", "Medium"), ("Medium", "High")]
n_bonferroni = len(pairs)
results = []
for a, b in pairs:
    if a not in groups or b not in groups:
        continue
    u, p = scipy_stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided")
    p_adj = min(p * n_bonferroni, 1.0)
    direction = "Higher" if groups[a].mean() < groups[b].mean() else "Lower"
    results.append({
        "pair": f"{a} vs {b}",
        "U statistic": round(u, 1),
        "p raw": round(p, 6),
        "p Bonferroni": round(p_adj, 6),
        "significant": p_adj < 0.05,
        f"{b} gen_std direction": direction,
    })
display(pd.DataFrame(results).set_index("pair"))

Group sizes: {'Low': 30, 'Medium': 141, 'High': 61}

Kruskal-Wallis H = 72.0952   p = 2.2116e-16
→ Significant difference across regimes (p < 0.05)

Pairwise Mann-Whitney U (Bonferroni-corrected α = 0.05/3):


,U statistic,p raw,p Bonferroni,significant,High gen_std direction,Medium gen_std direction
pair,,,,,,
Low vs High,137.0,0.0,0.000000,True,Higher,NaN
Low vs Medium,865.0,0.0,0.000001,True,NaN,Higher
Medium vs High,1797.0,0.0,0.000000,True,Higher,NaN


## 11. Sample-Level Std Spread Within Each Window

Beyond the mean std across samples, it is informative to look at the
**spread of stds across the 5 samples** for each window.  A model that has
genuinely learned the volatility regime should show low spread within a
window (all samples consistently more/less volatile) rather than noisy
disagreement.

In [25]:
df_spread = (
    df_gen
    .groupby(["file", "window_start", "regime"], observed=True)["gen_std_un"]
    .std()
    .rename("sample_spread")
    .reset_index()
)

print("Sample std spread (std of per-sample stds) by regime:")
display(
    df_spread.groupby("regime")["sample_spread"]
    .describe(percentiles=[.25, .5, .75])
    .round(6)
)

fig, ax = plt.subplots(figsize=(7, 4))
data_spread = [
    df_spread.loc[df_spread["regime"] == r, "sample_spread"].dropna().values
    for r in labels_present
]
bp2 = ax.boxplot(data_spread, labels=labels_present, patch_artist=True)
for patch, label in zip(bp2["boxes"], labels_present):
    patch.set_facecolor(REGIME_COLORS[label])
    patch.set_alpha(0.6)
ax.set_ylabel("Std of per-sample stds (unnorm)")
ax.set_title("Sample-to-sample spread of generated volatility by regime")
ax.set_xlabel("Reference volatility regime")
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"sample_spread_{appendix}.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

Sample std spread (std of per-sample stds) by regime:


,count,mean,std,min,25%,50%,75%,max
regime,,,,,,,,
High,61.0,0.004122,0.001494,0.000901,0.003079,0.003901,0.005012,0.008871
Low,30.0,0.002241,0.001324,0.000582,0.001544,0.001886,0.002352,0.007654
Medium,141.0,0.003554,0.001985,0.000608,0.002090,0.003315,0.004538,0.012769
Unknown,5.0,0.003987,0.002054,0.002260,0.002836,0.003016,0.004459,0.007365


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27152\3033501108.py:21: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_spread, labels=labels_present, patch_artist=True)


<Figure size 700x400 with 1 Axes>

## 12. Time-Series Overlay: Reference vs Generated Volatility

Plots `period_ref_std` (reference) and `gen_std_un` (generated) over
calendar time using each window's `start_date` as the x-coordinate.
Visual correlation between the two signals indicates the model tracks
the temporal volatility structure of the market.

In [26]:
fig, ax = plt.subplots(figsize=(14, 4))

# Reference signal (full calendar — thin line)
ax.plot(ts_sorted, mean_std_ts, lw=0.7, color="tab:blue", alpha=0.6,
        label="Reference mean_std (all windows)")

# Generated windows — scatter coloured by regime
for regime, grp in df_win_agg.groupby("regime", observed=True):
    ax.scatter(
        grp["start_date_ts"], grp["gen_std_un"],
        s=30, zorder=5, label=f"Generated — {regime}",
        color=REGIME_COLORS.get(regime, "grey"), alpha=0.8, edgecolors="none"
    )

ax.set_ylabel("Volatility (unnorm log-return std)")
ax.set_xlabel("Window start date")
ax.set_title("Reference volatility signal vs generated sample volatility over time")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"ts_overlay_{appendix}.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

<Figure size 1400x400 with 1 Axes>

## 13. Summary Table

Collects the key regime-level statistics in one place for easy reporting.

In [ ]:
summary_rows = []
for regime in REGIME_ORDER:
    grp = df_win_agg[df_win_agg["regime"] == regime]
    if grp.empty:
        continue
    ref_vals = grp["period_ref_std"].dropna()
    gen_vals = grp["gen_std_un"].dropna()
    summary_rows.append({
        "Regime"                  : regime,
        "N windows"               : len(grp),
        "Ref period_std mean"     : ref_vals.mean(),
        "Ref period_std median"   : ref_vals.median(),
        "Gen std mean"            : gen_vals.mean(),
        "Gen std median"          : gen_vals.median(),
        "Gen std std"             : gen_vals.std(),
        "Gen/Ref ratio (means)"   : gen_vals.mean() / ref_vals.mean() if ref_vals.mean() != 0 else np.nan,
    })

summary_df = pd.DataFrame(summary_rows).set_index("Regime")
print("=" * 70)
print("SUMMARY: Generated volatility by reference regime")
print("=" * 70)
display(summary_df.round(6))

# Interpretation note
if "Low" in groups and "High" in groups:
    low_mean  = groups["Low"].mean()
    high_mean = groups["High"].mean()
    ratio     = high_mean / low_mean if low_mean > 0 else np.nan
    direction = "HIGHER" if high_mean > low_mean else "LOWER"
    print(f"\nHigh-regime windows generate {direction} std than Low-regime "
          f"(ratio High/Low = {ratio:.3f}).")
    print("A ratio > 1 with a statistically significant KW test suggests "
          "the model has learned period-specific volatility.")

SUMMARY: Generated volatility by reference regime


,N windows,Ref period_std mean,Ref period_std median,Gen std mean,Gen std median,Gen std std,Gen/Ref ratio (means)
Regime,,,,,,,
Low,30,0.01381,0.013982,0.012119,0.011945,0.002037,0.877576
Medium,141,0.01763,0.017749,0.014022,0.013857,0.002236,0.795341
High,61,0.02318,0.022511,0.016948,0.016717,0.002947,0.731143



High-regime windows generate HIGHER std than Low-regime (ratio High/Low = 1.398).
A ratio > 1 with a statistically significant KW test suggests the model has learned period-specific volatility.


: 